# Modeling Win Probability for a Marth Player Against Fox

In [1]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
from pymc.math import dot
import pytensor.tensor as pt

# Set random seed
seed = 6420
rng = np.random.default_rng(seed)

In [2]:
# Load data
data = pd.read_csv("data/fox_marth_competitive.csv")

# Append synthetic elo data for each character (1000 - 3000)
data["fox_elo"] = rng.integers(1000, 3000, size=len(data))
data["marth_elo"] = rng.integers(1000, 3000, size=len(data))

# Count wins for each character
num_matches = len(data)
num_marth_wins = (data["winning_character"] == "MARTH").sum()
num_fox_wins = num_matches - num_marth_wins
print(f"Total matches: {num_matches}")
print(f"Matches won by Marth: {num_marth_wins}")
print(f"Matches won by Fox: {num_fox_wins}")

data = data.head(10000)  # Limit to first 10000 rows for faster sampling

# Show the first few rows of the data
print(data.head())

Total matches: 270974
Matches won by Marth: 133582
Matches won by Fox: 137392
   id               stage winning_character  fox_elo  marth_elo
0   1        YOSHIS_STORY               FOX     2885       1762
1   2  FOUNTAIN_OF_DREAMS             MARTH     1379       2230
2   3   FINAL_DESTINATION               FOX     1248       1164
3   4   FINAL_DESTINATION             MARTH     1757       2300
4   5        YOSHIS_STORY               FOX     1384       1396


In [3]:
stage_idx, stages = pd.factorize(data["stage"])
print("Unique stages:", stages)
print("Stage indices:", stage_idx)
print("Stage labels:", stages.tolist())
print("Number of unique stages:", len(stages))

Unique stages: Index(['YOSHIS_STORY', 'FOUNTAIN_OF_DREAMS', 'FINAL_DESTINATION',
       'POKEMON_STADIUM', 'BATTLEFIELD', 'DREAMLAND'],
      dtype='str')
Stage indices: [0 1 2 ... 5 5 1]
Stage labels: ['YOSHIS_STORY', 'FOUNTAIN_OF_DREAMS', 'FINAL_DESTINATION', 'POKEMON_STADIUM', 'BATTLEFIELD', 'DREAMLAND']
Number of unique stages: 6


In [4]:
elo_diff = (data["marth_elo"].values - data["fox_elo"].values) / 400.0
y = (data["winning_character"] == "MARTH").astype(int).values
print("Number of matches:", len(y))
print("Number of matches won by Marth:", y.sum())
print("Number of matches won by Fox:", len(y) - y.sum())

print("Number of yoshis:", np.sum(data["stage"] == "YOSHIS_STORY"))
print("Number won by Marth on Yoshi's Story:", np.sum((data["stage"] == "YOSHIS_STORY") & (data["winning_character"] == "MARTH")))
print("Number of matches on Battlefield:", np.sum(data["stage"] == "BATTLEFIELD"))
print("Number won by Marth on Battlefield:", np.sum((data["stage"] == "BATTLEFIELD") & (data["winning_character"] == "MARTH")))
print("Number of matches on Final Destination:", np.sum(data["stage"] == "FINAL_DESTINATION"))
print("Number won by Marth on Final Destination:", np.sum((data["stage"] == "FINAL_DESTINATION") & (data["winning_character"] == "MARTH")))

with pm.Model(coords={"stage": stages}) as model:

    # Elo scaling
    beta = pm.Normal("beta", mu=1.0, sigma=1.0)

    # Global matchup (Marth vs Fox)
    alpha = pm.Normal("alpha", mu=0.0, sigma=1.0)

    # Stage effects (ZERO-SUM CONSTRAINT)
    gamma = pm.ZeroSumNormal("gamma", sigma=0.5, dims="stage")

    # Linear predictor
    logit_p = beta * elo_diff + alpha + gamma[stage_idx]

    # Likelihood
    p = pm.math.sigmoid(logit_p)
    y_obs = pm.Bernoulli("y_obs", p=p, observed=y)

    trace = pm.sample(1000, tune=1000, target_accept=0.9, random_seed=seed)

Initializing NUTS using jitter+adapt_diag...


Number of matches: 10000
Number of matches won by Marth: 4298
Number of matches won by Fox: 5702
Number of yoshis: 1763
Number won by Marth on Yoshi's Story: 810
Number of matches on Battlefield: 1739
Number won by Marth on Battlefield: 739
Number of matches on Final Destination: 1570
Number won by Marth on Final Destination: 701


Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta, alpha, gamma]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 7 seconds.


In [5]:
az.summary(trace, var_names=["alpha", "beta", "gamma"])

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,-0.284,0.020,-0.320,-0.244,0.000,0.000,7056.0,3328.0,1.0
beta,0.003,0.010,-0.016,0.021,0.000,0.000,5472.0,2895.0,1.0
gamma[YOSHIS_STORY],0.120,0.044,0.038,0.199,0.001,0.001,7327.0,2623.0,1.0
gamma[FOUNTAIN_OF_DREAMS],-0.020,0.045,-0.101,0.063,0.001,0.001,5446.0,3178.0,1.0
gamma[FINAL_DESTINATION],0.068,0.045,-0.019,0.148,0.001,0.001,6203.0,3313.0,1.0
gamma[POKEMON_STADIUM],-0.081,0.045,-0.164,0.004,0.001,0.001,7743.0,3117.0,1.0
gamma[BATTLEFIELD],-0.018,0.045,-0.105,0.063,0.001,0.001,6118.0,2683.0,1.0
gamma[DREAMLAND],-0.069,0.049,-0.162,0.018,0.001,0.001,6031.0,2883.0,1.0


In [6]:
# Model with just stage effects
with pm.Model(coords={"stage": stages}) as stage_only_model:

    # Stage effects (ZERO-SUM CONSTRAINT)
    gamma = pm.ZeroSumNormal("gamma", sigma=0.5, dims="stage")

    # Matchup effect
    alpha = pm.Normal("alpha", mu=0.0, sigma=1.0)
    # alpha = pm.Beta("alpha", alpha=2.0, beta=2.0)  # Constrain to [0, 1]

    # Linear predictor
    logit_p = gamma[stage_idx] + alpha

    # Likelihood
    p = pm.math.sigmoid(logit_p)
    y_obs = pm.Bernoulli("y_obs", p=p, observed=y)

    stage_only_trace = pm.sample(1000, tune=1000, target_accept=0.9, random_seed=seed)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [gamma, alpha]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 6 seconds.


In [7]:
az.summary(stage_only_trace, hdi_prob=0.95)

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,-0.283,0.020,-0.323,-0.245,0.000,0.000,6260.0,2979.0,1.0
gamma[YOSHIS_STORY],0.121,0.045,0.036,0.207,0.001,0.001,4794.0,2987.0,1.0
gamma[FOUNTAIN_OF_DREAMS],-0.020,0.046,-0.113,0.066,0.001,0.001,6091.0,2785.0,1.0
gamma[FINAL_DESTINATION],0.068,0.046,-0.024,0.154,0.001,0.001,5309.0,2215.0,1.0
gamma[POKEMON_STADIUM],-0.081,0.044,-0.164,0.006,0.001,0.001,6355.0,3175.0,1.0
gamma[BATTLEFIELD],-0.019,0.043,-0.101,0.069,0.001,0.001,6510.0,3514.0,1.0
gamma[DREAMLAND],-0.069,0.045,-0.155,0.023,0.001,0.001,6085.0,3203.0,1.0
